# A6 – Text Intelligence: LLM Prototype Lab

**Project:** HealthDestination – Medical Services Finder  
**Task:** Given a user symptom description, classify severity as `non_emergency`, `moderate`, or `urgent`, and suggest an appropriate care setting.

---

## Section 0 – Install Dependencies

In [36]:
# Run this cell first to install/download everything needed.

%pip install -q transformers torch accelerate sentencepiece protobuf
%pip install -q pandas tabulate
%pip install -q groq python-dotenv   # Groq API for medium/large/XL models
%pip install -q google-generativeai  # Gemini API for XL models


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---

## Section 1 – Research & Categorization

Models were sourced from the [Hugging Face Open LLM Leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard) and filtered by:
- Publicly available weights (no gated/license-blocked models)
- Loadable via `transformers` pipeline
- Relevance to text classification / instruction following

### Model Categories

Large                       llama-3.3-70b-versatile      90%         0.27s

Large     meta-llama/llama-4-scout-17b-16e-instruct      85%         0.58s

Medium                                gemma-3-12b-it      85%        41.16s

Medium                                gemma-3-27b-it      85%         6.00s

Medium     meta-llama/llama-4-scout-17b-16e-instruct      85%         0.47s

Small                                 gemma-3-4b-it      85%         6.18s

Small                          llama-3.1-8b-instant      75%         0.26s

XL meta-llama/llama-4-maverick-17b-128e-instruct      95%         0.31s

XL                   moonshotai/kimi-k2-instruct      95%         1.57s

XL                       llama-3.3-70b-versatile      90%         0.28s

XL              moonshotai/kimi-k2-instruct-0905      85%         1.15s




---

## Section 2 – Fixed Test Inputs

These five symptom descriptions are used for every model to allow fair comparison.

In [26]:
TEST_INPUTS = [
    # ---------------- NON EMERGENCY ----------------
    "I have a slight headache and a runny nose. No fever.",
    "I twisted my ankle yesterday. It's swollen but I can walk on it.",
    "I have a mild fever of 99F and a sore throat for two days.",
    "My throat feels dry and scratchy but I can still eat normally.",
    "I feel a little tired and sleepy after a long week of work.",

    # ---------------- MODERATE ----------------
    "My ankle is swollen and painful when I walk.",
    "I have a fever of 101F and body aches.",
    "I've had vomiting and diarrhea since yesterday.",
    "My ear hurts and my hearing feels muffled.",
    "I have a cough that has lasted for a week.",

    # ---------------- URGENT ----------------
    "I've had chest pain and shortness of breath for the past hour. It feels tight.",
    "My face is drooping on one side and I can't raise my left arm. Started 10 minutes ago.",
    "I suddenly cannot breathe properly and feel dizzy.",
    "There is heavy bleeding from a deep cut in my leg.",
    "I suddenly lost vision in one eye.",

    # ---------------- EDGE CASES ----------------
    "I feel kind of weird today, maybe sick but not sure.",
    "My chest feels tight but it might just be anxiety.",
    "My head hurts so bad I can't see clearly.",
    "My stomach hurts a little after eating too much fast food.",
    "Pain level 9 out of 10 in my abdomen for three hours."
]

# Expected labels for reference (ground truth)
EXPECTED = [
    "non_emergency",
    "non_emergency",
    "non_emergency",
    "non_emergency",
    "non_emergency",

    "moderate",
    "moderate",
    "moderate",
    "moderate",
    "moderate",

    "urgent",
    "urgent",
    "urgent",
    "urgent",
    "urgent",

    "moderate",
    "moderate",
    "urgent",
    "non_emergency",
    "urgent"
]

CANDIDATE_LABELS = ["non_emergency", "moderate", "urgent"]

print(f"Loaded {len(TEST_INPUTS)} test cases.")

Loaded 20 test cases.


---

## Section 3 – The Matrix Test

### 3A – Ultra-Light Models (<1B) via Zero-Shot Classification Pipeline

These models use NLI (Natural Language Inference) to classify without any training examples.

In [3]:
import time
import pandas as pd
from transformers import pipeline

ULTRA_LIGHT_MODELS = [
    "distilbert-base-uncased",      # 66M – fast but weaker NLI
    "facebook/bart-large-mnli",     # 406M – strong NLI baseline
    "typeform/distilbert-base-uncased-mnli",  # 66M – distilbert fine-tuned on MNLI
]

results_ultra = []

for model_id in ULTRA_LIGHT_MODELS:
    print(f"\nLoading: {model_id}")
    try:
        clf = pipeline("zero-shot-classification", model=model_id)
        for i, text in enumerate(TEST_INPUTS):
            t0 = time.time()
            out = clf(text, candidate_labels=CANDIDATE_LABELS)
            elapsed = round(time.time() - t0, 2)
            predicted = out["labels"][0]
            correct = (predicted == EXPECTED[i])
            results_ultra.append({
                "model": model_id.split("/")[-1],
                "input_idx": i,
                "predicted": predicted,
                "expected": EXPECTED[i],
                "correct": correct,
                "latency_s": elapsed,
            })
            print(f"  [{i}] {predicted:15s} | expected={EXPECTED[i]:15s} | {'OK' if correct else 'WRONG'} | {elapsed}s")
        del clf
    except Exception as e:
        print(f"  ERROR: {e}")

df_ultra = pd.DataFrame(results_ultra)
print("\nUltra-Light summary:")
print(df_ultra.groupby("model")[["correct", "latency_s"]].agg({"correct": "mean", "latency_s": "mean"}))

/Users/jasonwu/Documents/INFO490/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading: distilbert-base-uncased


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12797.27it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Failed to determine 'entailment' label id from the label2id mapping in the model config. Setting to -1. Define a descriptive label2id mapping in the model config to

  [0] urgent          | expected=non_emergency   | WRONG | 2.58s
  [1] urgent          | expected=urgent          | OK | 0.31s
  [2] urgent          | expected=non_emergency   | WRONG | 0.06s
  [3] urgent          | expected=urgent          | OK | 0.17s
  [4] urgent          | expected=non_emergency   | WRONG | 0.05s

Loading: facebook/bart-large-mnli


Loading weights: 100%|██████████| 515/515 [00:00<00:00, 6221.01it/s]


  [0] non_emergency   | expected=non_emergency   | OK | 6.95s
  [1] urgent          | expected=urgent          | OK | 0.19s
  [2] moderate        | expected=non_emergency   | WRONG | 0.16s
  [3] urgent          | expected=urgent          | OK | 1.13s
  [4] moderate        | expected=non_emergency   | WRONG | 0.13s

Loading: typeform/distilbert-base-uncased-mnli


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 10632.99it/s]


  [0] moderate        | expected=non_emergency   | WRONG | 0.13s
  [1] moderate        | expected=urgent          | WRONG | 0.04s
  [2] urgent          | expected=non_emergency   | WRONG | 0.04s
  [3] moderate        | expected=urgent          | WRONG | 0.04s
  [4] moderate        | expected=non_emergency   | WRONG | 0.03s

Ultra-Light summary:
                              correct  latency_s
model                                           
bart-large-mnli                   0.6      1.712
distilbert-base-uncased           0.4      0.634
distilbert-base-uncased-mnli      0.0      0.056


### 3B–3E – Small / Medium / Large / XL Models via Groq API

Local inference for models ≥1B parameters is impractical on CPU (1.1B TinyLlama took 200s/input in testing).
We use [Groq's free API](https://console.groq.com) instead — it serves the same open-weight models (Llama, Mistral, Gemma) with sub-second latency.




In [38]:
import os, time
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

load_dotenv(dotenv_path="../.env")   # loads GROQ_API_KEY from project .env
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    print("ERROR: GROQ_API_KEY not set. Add it to your .env file as: GROQ_API_KEY=gsk_...")
else:
    groq_client = Groq(api_key=GROQ_API_KEY)
    print("Groq client ready.")

def build_prompt_zeroshot(symptom_text):
    return (
        "Classify the medical severity of the following symptom description into "
        "exactly one word: non_emergency, moderate, or urgent.\n\n"
        f"Symptoms: {symptom_text}\n\n"
        "Severity (one word only):"
    )

def extract_label(raw_output: str) -> str:
    raw = raw_output.lower()
    if "urgent" in raw:
        return "urgent"
    if "moderate" in raw:
        return "moderate"
    if "non_emergency" in raw or "non emergency" in raw or "non-emergency" in raw:
        return "non_emergency"
    return "unknown"

def groq_classify(model_id, symptom_text):
    prompt = build_prompt_zeroshot(symptom_text)
    resp = groq_client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=15,
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

print("Helper functions ready.")

Groq client ready.
Helper functions ready.


In [43]:
import google.generativeai as genai

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    print("WARNING: GEMINI_API_KEY not set. Add it to your .env file.")
else:
    genai.configure(api_key=GEMINI_API_KEY)
    print("Gemini client ready.")

def gemini_classify(model_id, symptom_text):
    prompt = build_prompt_zeroshot(symptom_text)

    model = genai.GenerativeModel(model_id)

    resp = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            temperature=0,
            max_output_tokens=10
        )
    )

    return resp.text.strip()

Gemini client ready.


### 3B – Small Models (1B–3B) via Groq

In [44]:
# NOTE: All sub-8B Groq text models (llama-3.2-1b-preview, llama-3.2-3b-preview) were
# deprecated on 04/14/25. llama-3.1-8b-instant is the smallest text model currently available.
SMALL_GROQ_MODELS = [
    ("llama-3.1-8b-instant", 8000, "groq"),
    ("gemma-3-4b-it", 4000, "gemini"),
]

results_small = []

for model_id, params_M, provider in SMALL_GROQ_MODELS:
    label = f"{model_id} ({provider})"
    print(f"\nModel: {label}")
    for i, text in enumerate(TEST_INPUTS):
        t0 = time.time()
        try:
            if provider == "groq":
                raw = groq_classify(model_id, text)
            else:
                raw = gemini_classify(model_id, text)
                time.sleep(5)  # free-tier rate limit protection
            elapsed = round(time.time() - t0, 2)
            predicted = extract_label(raw)
            correct = (predicted == EXPECTED[i])
            results_small.append({
                "model": model_id, "params_M": params_M,
                "input_idx": i, "raw_output": raw[:50],
                "predicted": predicted, "expected": EXPECTED[i],
                "correct": correct, "latency_s": elapsed,
            })
            print(f"  [{i}] {predicted:15s} | expected={EXPECTED[i]:15s} | {'OK' if correct else 'WRONG'} | {elapsed}s")
        except Exception as e:
            print(f"  [{i}] ERROR: {e}")

df_small = pd.DataFrame(results_small)
if not df_small.empty:
    print("\nSmall model summary:")
    print(df_small.groupby("model")[["correct", "latency_s"]].agg({"correct": "mean", "latency_s": "mean"}))


Model: llama-3.1-8b-instant (groq)
  [0] non_emergency   | expected=non_emergency   | OK | 0.6s
  [1] moderate        | expected=non_emergency   | WRONG | 0.2s
  [2] moderate        | expected=non_emergency   | WRONG | 0.19s
  [3] moderate        | expected=non_emergency   | WRONG | 0.42s
  [4] non_emergency   | expected=non_emergency   | OK | 0.18s
  [5] moderate        | expected=moderate        | OK | 0.22s
  [6] moderate        | expected=moderate        | OK | 0.22s
  [7] moderate        | expected=moderate        | OK | 0.23s
  [8] moderate        | expected=moderate        | OK | 0.21s
  [9] moderate        | expected=moderate        | OK | 0.38s
  [10] urgent          | expected=urgent          | OK | 0.19s
  [11] urgent          | expected=urgent          | OK | 0.18s
  [12] urgent          | expected=urgent          | OK | 0.21s
  [13] urgent          | expected=urgent          | OK | 0.27s
  [14] urgent          | expected=urgent          | OK | 0.19s
  [15] non_emergency  

### 3C – Medium Models (3B–7B) via Groq

In [45]:
MEDIUM_GROQ_MODELS = [
    ("meta-llama/llama-4-scout-17b-16e-instruct", 17000, "groq"),  # replaces llama-3.3-70b-specdec (deprecated 04/14/25)
    ("gemma-3-12b-it", 12000, "gemini"),
    ("gemma-3-27b-it", 27000, "gemini"),
]

results_medium = []

for model_id, params_M, provider in MEDIUM_GROQ_MODELS:
    label = f"{model_id} ({provider})"
    print(f"\nModel: {label}")
    for i, text in enumerate(TEST_INPUTS):
        t0 = time.time()
        try:
            if provider == "groq":
                raw = groq_classify(model_id, text)
            else:
                raw = gemini_classify(model_id, text)
                time.sleep(5)  # free-tier rate limit protection
            elapsed = round(time.time() - t0, 2)
            predicted = extract_label(raw)
            correct = (predicted == EXPECTED[i])
            results_medium.append({
                "model": model_id, "params_M": params_M,
                "input_idx": i, "raw_output": raw[:50],
                "predicted": predicted, "expected": EXPECTED[i],
                "correct": correct, "latency_s": elapsed,
            })
            print(f"  [{i}] {predicted:15s} | expected={EXPECTED[i]:15s} | {'OK' if correct else 'WRONG'} | {elapsed}s")
        except Exception as e:
            print(f"  [{i}] ERROR: {e}")

df_medium = pd.DataFrame(results_medium)
if not df_medium.empty:
    print("\nMedium model summary:")
    print(df_medium.groupby("model")[["correct", "latency_s"]].agg({"correct": "mean", "latency_s": "mean"}))


Model: meta-llama/llama-4-scout-17b-16e-instruct (groq)
  [0] non_emergency   | expected=non_emergency   | OK | 0.56s
  [1] moderate        | expected=non_emergency   | WRONG | 0.44s
  [2] non_emergency   | expected=non_emergency   | OK | 0.39s
  [3] non_emergency   | expected=non_emergency   | OK | 0.36s
  [4] non_emergency   | expected=non_emergency   | OK | 0.5s
  [5] moderate        | expected=moderate        | OK | 0.43s
  [6] moderate        | expected=moderate        | OK | 0.72s
  [7] moderate        | expected=moderate        | OK | 0.19s
  [8] moderate        | expected=moderate        | OK | 0.41s
  [9] non_emergency   | expected=moderate        | WRONG | 1.11s
  [10] urgent          | expected=urgent          | OK | 0.28s
  [11] urgent          | expected=urgent          | OK | 0.48s
  [12] urgent          | expected=urgent          | OK | 0.47s
  [13] urgent          | expected=urgent          | OK | 0.43s
  [14] urgent          | expected=urgent          | OK | 0.62s
  [

### 3D – Large Models (7B–13B) via Groq

In [49]:
LARGE_GROQ_MODELS = [
    ("llama-3.3-70b-versatile",                 70000, "groq"),  # confirmed working
    ("meta-llama/llama-4-scout-17b-16e-instruct", 17000, "groq"),  # replaces llama3-70b-8192
]

results_large = []

for model_id, params_M, provider in LARGE_GROQ_MODELS:
    label = f"{model_id} ({provider})"
    print(f"\nModel: {label}")
    for i, text in enumerate(TEST_INPUTS):
        t0 = time.time()
        try:
            if provider == "groq":
                raw = groq_classify(model_id, text)
            else:
                for i, text in enumerate(TEST_INPUTS):
                    try:
                        raw = gemini_classify(model_id, text)
                        predicted = extract_label(raw)
                        # store results...
                    except Exception as e:
                        print(f"ERROR: {e}")
                    
                    # Wait to avoid hitting rate limits
                    time.sleep(6)  
            elapsed = round(time.time() - t0, 2)
            predicted = extract_label(raw)
            correct = (predicted == EXPECTED[i])
            results_large.append({
                "model": model_id, "params_M": params_M,
                "input_idx": i, "raw_output": raw[:50],
                "predicted": predicted, "expected": EXPECTED[i],
                "correct": correct, "latency_s": elapsed,
            })
            print(f"  [{i}] {predicted:15s} | expected={EXPECTED[i]:15s} | {'OK' if correct else 'WRONG'} | {elapsed}s")
        except Exception as e:
            print(f"  [{i}] ERROR: {e}")

df_large = pd.DataFrame(results_large)
if not df_large.empty:
    print("\nLarge model summary:")
    print(df_large.groupby("model")[["correct", "latency_s"]].agg({"correct": "mean", "latency_s": "mean"}))


Model: llama-3.3-70b-versatile (groq)
  [0] non_emergency   | expected=non_emergency   | OK | 0.61s
  [1] moderate        | expected=non_emergency   | WRONG | 0.21s
  [2] non_emergency   | expected=non_emergency   | OK | 0.39s
  [3] non_emergency   | expected=non_emergency   | OK | 0.2s
  [4] non_emergency   | expected=non_emergency   | OK | 0.31s
  [5] moderate        | expected=moderate        | OK | 0.2s
  [6] moderate        | expected=moderate        | OK | 0.18s
  [7] moderate        | expected=moderate        | OK | 0.29s
  [8] moderate        | expected=moderate        | OK | 0.47s
  [9] moderate        | expected=moderate        | OK | 0.21s
  [10] urgent          | expected=urgent          | OK | 0.24s
  [11] urgent          | expected=urgent          | OK | 0.19s
  [12] urgent          | expected=urgent          | OK | 0.38s
  [13] urgent          | expected=urgent          | OK | 0.2s
  [14] urgent          | expected=urgent          | OK | 0.18s
  [15] non_emergency   | e

### Checking Avaliable Grok Models

In [53]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'groq/compound-mini', 'object': 'model', 'created': 1756949707, 'owned_by': 'Groq', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'canopylabs/orpheus-v1-english', 'object': 'model', 'created': 1766186316, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000}, {'id': 'groq/compound', 'object': 'model', 'created': 1756949530, 'owned_by': 'Groq', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'meta-llama/llama-4-scout-17b-16e-instruct', 'object': 'model', 'created': 1743874824, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'meta-llama/llama-guard-4-12b', 'object': 'model', 'created': 1746743847, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 1024

### XL Models

In [59]:
XL_GROQ_MODELS = [
    ("meta-llama/llama-4-maverick-17b-128e-instruct", 17000, "groq"),
    ("llama-3.3-70b-versatile",                       70000, "groq"),
    ("moonshotai/kimi-k2-instruct",                   None,  "groq"),
    ("moonshotai/kimi-k2-instruct-0905",              None,  "groq")
]

results_xl = []

for model_id, params_M, provider in XL_GROQ_MODELS:
    label = f"{model_id} ({'Groq' if provider == 'groq' else 'Gemini'})"
    print(f"\nModel: {label}")
    for i, text in enumerate(TEST_INPUTS):
        t0 = time.time()
        try:
            if provider == "groq":
                raw = groq_classify(model_id, text)
            else:
                raw = gemini_classify(model_id, text)
            elapsed = round(time.time() - t0, 2)
            predicted = extract_label(raw)
            correct = (predicted == EXPECTED[i])
            results_xl.append({
                "model": model_id, "params_M": params_M,
                "input_idx": i, "raw_output": raw[:50],
                "predicted": predicted, "expected": EXPECTED[i],
                "correct": correct, "latency_s": elapsed,
            })
            print(f"  [{i}] {predicted:15s} | expected={EXPECTED[i]:15s} | {'OK' if correct else 'WRONG'} | {elapsed}s")
        except Exception as e:
            print(f"  [{i}] ERROR (model may not be on free tier): {e}")

df_xl = pd.DataFrame(results_xl)
if not df_xl.empty:
    print("\nXL model summary:")
    print(df_xl.groupby("model")[["correct", "latency_s"]].agg({"correct": "mean", "latency_s": "mean"}))


Model: meta-llama/llama-4-maverick-17b-128e-instruct (Groq)
  [0] non_emergency   | expected=non_emergency   | OK | 0.82s
  [1] moderate        | expected=non_emergency   | WRONG | 0.2s
  [2] non_emergency   | expected=non_emergency   | OK | 0.4s
  [3] non_emergency   | expected=non_emergency   | OK | 0.36s
  [4] non_emergency   | expected=non_emergency   | OK | 0.28s
  [5] moderate        | expected=moderate        | OK | 0.2s
  [6] moderate        | expected=moderate        | OK | 0.42s
  [7] moderate        | expected=moderate        | OK | 0.23s
  [8] moderate        | expected=moderate        | OK | 0.4s
  [9] moderate        | expected=moderate        | OK | 0.21s
  [10] urgent          | expected=urgent          | OK | 0.24s
  [11] urgent          | expected=urgent          | OK | 0.23s
  [12] urgent          | expected=urgent          | OK | 0.2s
  [13] urgent          | expected=urgent          | OK | 0.2s
  [14] urgent          | expected=urgent          | OK | 0.29s
  [15] 

### First Best Model Meta-Lllama 17b-128e-instruct

In [63]:
BEST_MODEL = "meta-llama/llama-4-maverick-17b-128e-instruct"  

def groq_prompt(model_id, prompt_text):
    resp = groq_client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt_text}],
        max_tokens=15,
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

# ── Zero-Shot ──────────────────────────────────────────────────────────────
def prompt_zero_shot(symptoms):
    return (
        "Classify the medical urgency of the following symptoms as exactly one word: "
        "non_emergency, moderate, or urgent.\n\n"
        f"Symptoms: {symptoms}\n"
        "Severity:"
    )

# ── One-Shot ───────────────────────────────────────────────────────────────
def prompt_one_shot(symptoms):
    return (
        "Classify the medical urgency of the following symptoms as exactly one word: "
        "non_emergency, moderate, or urgent.\n\n"
        "Example:\n"
        "Symptoms: I have a mild cold with sneezing.\n"
        "Severity: non_emergency\n\n"
        f"Symptoms: {symptoms}\n"
        "Severity:"
    )

# ── Few-Shot ───────────────────────────────────────────────────────────────
def prompt_few_shot(symptoms):
    return (
        "Classify the medical urgency of the following symptoms as exactly one word: "
        "non_emergency, moderate, or urgent.\n\n"
        "Examples:\n"
        "Symptoms: I have a mild cold with sneezing.\n"
        "Severity: non_emergency\n\n"
        "Symptoms: I twisted my knee, it is swollen and painful to walk.\n"
        "Severity: moderate\n\n"
        "Symptoms: I have sudden severe chest pain radiating to my left arm.\n"
        "Severity: urgent\n\n"
        f"Symptoms: {symptoms}\n"
        "Severity:"
    )

# ── Many-Shot ──────────────────────────────────────────────────────────────
def prompt_many_shot(symptoms):
    return (
        "Classify the medical urgency of the following symptoms as exactly one word: "
        "non_emergency, moderate, or urgent.\n\n"
        "Examples:\n"
        "Symptoms: I have a mild cold with sneezing.\nSeverity: non_emergency\n\n"
        "Symptoms: I have a low-grade fever of 99F and a runny nose.\nSeverity: non_emergency\n\n"
        "Symptoms: I cut my finger slightly while cooking.\nSeverity: non_emergency\n\n"
        "Symptoms: I twisted my knee, it is swollen and painful to walk.\nSeverity: moderate\n\n"
        "Symptoms: I have a persistent headache for two days with mild nausea.\nSeverity: moderate\n\n"
        "Symptoms: I burned my hand, it is blistering.\nSeverity: moderate\n\n"
        "Symptoms: I have sudden severe chest pain radiating to my left arm.\nSeverity: urgent\n\n"
        "Symptoms: My face is drooping and I cannot lift my arm.\nSeverity: urgent\n\n"
        "Symptoms: I am having difficulty breathing and my lips are turning blue.\nSeverity: urgent\n\n"
        f"Symptoms: {symptoms}\n"
        "Severity:"
    )

PROMPT_STRATEGIES = {
    "zero_shot": prompt_zero_shot,
    "one_shot":  prompt_one_shot,
    "few_shot":  prompt_few_shot,
    "many_shot": prompt_many_shot,
}

print("Prompt strategies defined.")

Prompt strategies defined.


In [64]:
results_prompt = []

for strategy_name, prompt_fn in PROMPT_STRATEGIES.items():
    print(f"\n--- Strategy: {strategy_name} ---")
    correct_count = 0
    for i, text in enumerate(TEST_INPUTS):
        prompt = prompt_fn(text)
        t0 = time.time()
        try:
            raw = groq_prompt(BEST_MODEL, prompt)
            elapsed = round(time.time() - t0, 2)
            predicted = extract_label(raw)
            correct = (predicted == EXPECTED[i])
            if correct:
                correct_count += 1
            results_prompt.append({
                "strategy": strategy_name,
                "input_idx": i,
                "input_text": text[:40] + "...",
                "raw_output": raw[:50],
                "predicted": predicted,
                "expected": EXPECTED[i],
                "correct": correct,
                "latency_s": elapsed,
            })
            print(f"  [{i}] {predicted:15s} | expected={EXPECTED[i]:15s} | {'OK' if correct else 'WRONG'} | raw='{raw[:30]}'")
        except Exception as e:
            print(f"  [{i}] ERROR: {e}")
    acc = correct_count / len(TEST_INPUTS)
    print(f"  Accuracy: {acc:.0%}")

df_prompt = pd.DataFrame(results_prompt)

print("\n=== Prompt Strategy Comparison ===")
summary = df_prompt.groupby("strategy")["correct"].agg(["mean", "sum"])
summary.columns = ["accuracy", "correct_count"]
summary["accuracy"] = summary["accuracy"].map("{:.0%}".format)
print(summary)


--- Strategy: zero_shot ---
  [0] non_emergency   | expected=non_emergency   | OK | raw='non_emergency'
  [1] moderate        | expected=non_emergency   | WRONG | raw='moderate'
  [2] non_emergency   | expected=non_emergency   | OK | raw='non_emergency'
  [3] non_emergency   | expected=non_emergency   | OK | raw='non_emergency'
  [4] non_emergency   | expected=non_emergency   | OK | raw='non_emergency'
  [5] moderate        | expected=moderate        | OK | raw='Moderate. 

So, the one-word c'
  [6] moderate        | expected=moderate        | OK | raw='Moderate. 

So the answer is: '
  [7] moderate        | expected=moderate        | OK | raw='Moderate. 

So the answer is: '
  [8] moderate        | expected=moderate        | OK | raw='Moderate. 

So the response is'
  [9] moderate        | expected=moderate        | OK | raw='Moderate. 

So the answer is: '
  [10] urgent          | expected=urgent          | OK | raw='Urgent.'
  [11] urgent          | expected=urgent          | OK | 

### Second Best Model Moonshot Kimi K2-Instruct

In [60]:
BEST_MODEL = "moonshotai/kimi-k2-instruct"  

def groq_prompt(model_id, prompt_text):
    resp = groq_client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt_text}],
        max_tokens=15,
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

# ── Zero-Shot ──────────────────────────────────────────────────────────────
def prompt_zero_shot(symptoms):
    return (
        "Classify the medical urgency of the following symptoms as exactly one word: "
        "non_emergency, moderate, or urgent.\n\n"
        f"Symptoms: {symptoms}\n"
        "Severity:"
    )

# ── One-Shot ───────────────────────────────────────────────────────────────
def prompt_one_shot(symptoms):
    return (
        "Classify the medical urgency of the following symptoms as exactly one word: "
        "non_emergency, moderate, or urgent.\n\n"
        "Example:\n"
        "Symptoms: I have a mild cold with sneezing.\n"
        "Severity: non_emergency\n\n"
        f"Symptoms: {symptoms}\n"
        "Severity:"
    )

# ── Few-Shot ───────────────────────────────────────────────────────────────
def prompt_few_shot(symptoms):
    return (
        "Classify the medical urgency of the following symptoms as exactly one word: "
        "non_emergency, moderate, or urgent.\n\n"
        "Examples:\n"
        "Symptoms: I have a mild cold with sneezing.\n"
        "Severity: non_emergency\n\n"
        "Symptoms: I twisted my knee, it is swollen and painful to walk.\n"
        "Severity: moderate\n\n"
        "Symptoms: I have sudden severe chest pain radiating to my left arm.\n"
        "Severity: urgent\n\n"
        f"Symptoms: {symptoms}\n"
        "Severity:"
    )

# ── Many-Shot ──────────────────────────────────────────────────────────────
def prompt_many_shot(symptoms):
    return (
        "Classify the medical urgency of the following symptoms as exactly one word: "
        "non_emergency, moderate, or urgent.\n\n"
        "Examples:\n"
        "Symptoms: I have a mild cold with sneezing.\nSeverity: non_emergency\n\n"
        "Symptoms: I have a low-grade fever of 99F and a runny nose.\nSeverity: non_emergency\n\n"
        "Symptoms: I cut my finger slightly while cooking.\nSeverity: non_emergency\n\n"
        "Symptoms: I twisted my knee, it is swollen and painful to walk.\nSeverity: moderate\n\n"
        "Symptoms: I have a persistent headache for two days with mild nausea.\nSeverity: moderate\n\n"
        "Symptoms: I burned my hand, it is blistering.\nSeverity: moderate\n\n"
        "Symptoms: I have sudden severe chest pain radiating to my left arm.\nSeverity: urgent\n\n"
        "Symptoms: My face is drooping and I cannot lift my arm.\nSeverity: urgent\n\n"
        "Symptoms: I am having difficulty breathing and my lips are turning blue.\nSeverity: urgent\n\n"
        f"Symptoms: {symptoms}\n"
        "Severity:"
    )

PROMPT_STRATEGIES = {
    "zero_shot": prompt_zero_shot,
    "one_shot":  prompt_one_shot,
    "few_shot":  prompt_few_shot,
    "many_shot": prompt_many_shot,
}

print("Prompt strategies defined.")

Prompt strategies defined.


In [62]:
results_prompt = []

for strategy_name, prompt_fn in PROMPT_STRATEGIES.items():
    print(f"\n--- Strategy: {strategy_name} ---")
    correct_count = 0
    for i, text in enumerate(TEST_INPUTS):
        prompt = prompt_fn(text)
        t0 = time.time()
        try:
            raw = groq_prompt(BEST_MODEL, prompt)
            elapsed = round(time.time() - t0, 2)
            predicted = extract_label(raw)
            correct = (predicted == EXPECTED[i])
            if correct:
                correct_count += 1
            results_prompt.append({
                "strategy": strategy_name,
                "input_idx": i,
                "input_text": text[:40] + "...",
                "raw_output": raw[:50],
                "predicted": predicted,
                "expected": EXPECTED[i],
                "correct": correct,
                "latency_s": elapsed,
            })
            print(f"  [{i}] {predicted:15s} | expected={EXPECTED[i]:15s} | {'OK' if correct else 'WRONG'} | raw='{raw[:30]}'")
        except Exception as e:
            print(f"  [{i}] ERROR: {e}")
    acc = correct_count / len(TEST_INPUTS)
    print(f"  Accuracy: {acc:.0%}")

df_prompt = pd.DataFrame(results_prompt)

print("\n=== Prompt Strategy Comparison ===")
summary = df_prompt.groupby("strategy")["correct"].agg(["mean", "sum"])
summary.columns = ["accuracy", "correct_count"]
summary["accuracy"] = summary["accuracy"].map("{:.0%}".format)
print(summary)


--- Strategy: zero_shot ---
  [0] non_emergency   | expected=non_emergency   | OK | raw='non_emergency'
  [1] non_emergency   | expected=non_emergency   | OK | raw='non_emergency'
  [2] non_emergency   | expected=non_emergency   | OK | raw='non_emergency'
  [3] non_emergency   | expected=non_emergency   | OK | raw='non_emergency'
  [4] non_emergency   | expected=non_emergency   | OK | raw='non_emergency'
  [5] moderate        | expected=moderate        | OK | raw='moderate'
  [6] moderate        | expected=moderate        | OK | raw='moderate'
  [7] moderate        | expected=moderate        | OK | raw='moderate'
  [8] moderate        | expected=moderate        | OK | raw='moderate'
  [9] non_emergency   | expected=moderate        | WRONG | raw='non_emergency'
  [10] urgent          | expected=urgent          | OK | raw='urgent'
  [11] urgent          | expected=urgent          | OK | raw='urgent'
  [12] urgent          | expected=urgent          | OK | raw='urgent'
  [13] urgent     

In [65]:
all_frames = []

datasets = [
    ("Ultra-Light", globals().get("df_ultra")),
    ("Small", globals().get("df_small")),
    ("Medium", globals().get("df_medium")),
    ("Large", globals().get("df_large")),
    ("XL", globals().get("df_xl")),
]

for cat, df in datasets:
    if df is not None and not df.empty:
        df2 = df[["model", "correct", "latency_s"]].copy()
        df2["category"] = cat
        all_frames.append(df2)

if all_frames:
    combined = pd.concat(all_frames)

    summary_all = (
        combined
        .groupby(["category", "model"])
        .agg(
            accuracy=("correct", "mean"),
            avg_latency_s=("latency_s", "mean")
        )
        .reset_index()
        .sort_values(["category", "accuracy"], ascending=[True, False])
    )

    summary_all["accuracy"] = summary_all["accuracy"].map("{:.0%}".format)
    summary_all["avg_latency_s"] = summary_all["avg_latency_s"].map("{:.2f}s".format)

    print(summary_all.to_string(index=False))
else:
    print("No results yet — run the model cells above first.")

category                                         model accuracy avg_latency_s
   Large                       llama-3.3-70b-versatile      90%         0.27s
   Large     meta-llama/llama-4-scout-17b-16e-instruct      85%         0.58s
  Medium                                gemma-3-12b-it      85%        41.16s
  Medium                                gemma-3-27b-it      85%         6.00s
  Medium     meta-llama/llama-4-scout-17b-16e-instruct      85%         0.47s
   Small                                 gemma-3-4b-it      85%         6.18s
   Small                          llama-3.1-8b-instant      75%         0.26s
      XL meta-llama/llama-4-maverick-17b-128e-instruct      95%         0.31s
      XL                   moonshotai/kimi-k2-instruct      95%         1.57s
      XL                       llama-3.3-70b-versatile      90%         0.28s
      XL              moonshotai/kimi-k2-instruct-0905      85%         1.15s


### Evaluation Criteria Summary

| Criterion | Weight | Notes |
|-----------|--------|-------|
| **Instruction adherence** (outputs valid label) | 40% | Does the model return exactly one of the three labels? |
| **Accuracy** (matches expected label) | 40% | Correct classification of our 5 test cases |
| **Latency** (seconds per input) | 10% | Critical for a real-time Django view |
| **Memory footprint** | 10% | Can it run on a server without a GPU? |

**Chosen model for Django integration:** *Meta llama 4 Maverick 17b 128e-Instruct Model*  
**Reason:** *Highest Acurracy and lowest latency from 20 test cases + shot testings*